In [1]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import tiktoken
from openai import OpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core import document_loaders
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage


In [2]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("gemini_key start with:", gemini_api_key[:8])

chroma_api=os.getenv('CHROMA_API_KEY')

gemini_key start with: AIzaSyBY


In [3]:
import chromadb

client = chromadb.CloudClient(
  api_key=chroma_api,
  tenant='246553cc-048a-47fa-aa2a-cf9d61280656',
  database='Project'
)

In [4]:
import pdfplumber
from langchain_core.documents import Document

In [5]:
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

In [6]:
def add_text(text,college_id,client):
    docs=[]
    college_collection = f"college_{college_id}"
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    if text:
        docs.append(Document(page_content=text, metadata={"source": college_id, "college_id": college_collection}))

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)
    
    
    vectorestore = Chroma.from_documents(documents=chunks, embedding=embeddings, client=client, collection_name=college_collection)    
    
    print(f"Vector store created with {vectorestore._collection.count()} documents")
    return 

In [7]:
def extract_text_from_pdf(pdf_path, college_id, client):
    docs = []
    text = ""
    college_collection = f"college_{college_id}"
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore1 = Chroma(
        client=client,
        collection_name=college_collection,
        embedding_function=embeddings
    )
    exists = vectorstore1._collection.get(
        where={"source": pdf_path},
        limit=1
    )

    if exists["ids"]:
        print(f"'{pdf_path}' already exists in college_{college_id}")
        return
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, pages in enumerate(pdf.pages):
            text += pages.extract_text()
    if len(text.strip()) < 50:
        print("Falling back to ocr...")
        images=convert_from_path(pdf_path,dpi=300)
        ocr_text=[]
        for image in images:
            ocr_text.append(pytesseract.image_to_string(image,lang='eng'))    
        text="\n".join(ocr_text)    
            
    if text:
        docs.append(Document(page_content=text, metadata={"source": pdf_path, "college_id": college_collection}))   
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    
    
    vectorestore = Chroma.from_documents(documents=chunks, embedding=embeddings, client=client, collection_name=college_collection)    
    
    print(f"Vector store created with {vectorestore._collection.count()} documents")
    return 
               


In [9]:
extract_text_from_pdf("academic calendar.pdf",'pict',client)

Falling back to ocr...
Vector store created with 12 documents


In [8]:
def get_college_retriever(college_id, client, k=5):
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vectorstore = Chroma(
        client=client,
        collection_name=f"college_{college_id}",
        embedding_function=embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    return retriever
   



In [9]:
pict_retriever=get_college_retriever('pict',client)

In [10]:
context=pict_retriever.invoke("When will college commence?")
context

[Document(id='eb579e2d-c8f1-4361-aef7-77f7dadb468d', metadata={'college_id': 'college_pict', 'source': 'BNY_Mellon_Tech_Stack.pdf'}, page_content='BNY Mellon Tech Stack\nFrontend:\n- Web: React.js, Angular, HTML5, CSS3, JavaScript\n- Mobile: iOS (Swift, Objective-C), Android (Java, Kotlin)\nBackend:\n- Java, .NET (C#), Python\n- Frameworks: Spring Boot, ASP.NET Core\nDatabases:\n- Relational: Oracle, SQL Server, MySQL\n- NoSQL: MongoDB, Cassandra\n- Caching: Redis\nMessaging & Streaming:\n- Kafka, RabbitMQ, IBM MQ\nCloud & Infrastructure:\n- AWS, Microsoft Azure\n- Docker, Kubernetes\n- Terraform, Ansible\nMonitoring & Analytics:\n- Prometheus, Grafana, ELK Stack (Elasticsearch, Logstash, Kibana)\n- Splunk\nSecurity & Compliance:\n- PCI DSS, SOC2 compliant systems\n- Tokenization, Encryption protocols, IAM solutions\nDevOps & CI/CD:- Jenkins, GitLab CI/CD, Azure DevOps\n- Maven, Gradle, NuGet\nThis tech stack enables BNY Mellon to manage high-volume financial transactions securely and\

In [10]:
MODEL = "gemini-2.5-flash"


llm=ChatGoogleGenerativeAI(model=MODEL,temperature=0)

In [30]:
llm.invoke("When will the college start?")

AIMessage(content='That\'s a great question, but the answer **depends entirely on the specific college and its academic calendar!**\n\nHowever, I can give you some general information:\n\n1.  **Most Common (Fall Semester):** For the majority of colleges and universities in the United States, the main academic year (Fall semester) typically starts in **late August or early September**.\n2.  **Spring Semester:** Many colleges also have a Spring semester, which usually begins in **January**.\n3.  **Summer Sessions:** Summer sessions vary widely but generally run from **May through August**.\n4.  **Quarter System:** Some colleges operate on a quarter system instead of semesters. Their Fall quarter might start in late September, with Winter and Spring quarters following.\n5.  **International Colleges:** Start dates vary significantly by country. For example, many universities in the UK and Europe start in September or October.\n\n**To find the exact start date for *your* college, you should

In [11]:
SYSTEM_PREVIEW_MSG="""You are a helpful,friendly assistant for students to get their college notices.
You are chatting with students.
Use relevant context to generate response
If you don't know, say so.
Relevant context:
{context}
"""

In [14]:
def question_answer(query,history, college_name):
    
    retriever = get_college_retriever(college_name, client)
    rewritten_query=query_rewriting(query,history)
    print(rewritten_query)
    context = retriever.invoke(rewritten_query)
    print(context)
    text = "\n\n".join(doc.page_content for doc in context)
    system_prompt = SYSTEM_PREVIEW_MSG.format(context=text)

    msg=[SystemMessage(content=system_prompt),
    *gradio_history_to_messages(history),
    HumanMessage(content=query)]
    
    
    response = llm.invoke(msg)
    history.append({'role':'user',"content":query})
    history.append({'role':'assistant',"content":response.content})
    
    return history


In [15]:
history=question_answer("When will college commence?",[],'pict')

response1,history1=question_answer("give frontend bny stack",history,'pict')
history1



When will college commence?
[Document(id='393c6f8e-f031-4172-ab8d-172819702115', metadata={'source': 'pict', 'college_id': 'college_pict'}, page_content='The collge authorities have decided that the college for academic year 2025-26 will start from Monday 15th July, 2025.All the students keep note of this.'), Document(id='b6494af3-8577-43f5-a84c-2695494209a4', metadata={'college_id': 'college_pict', 'source': 'academic calendar.pdf'}, page_content='Sil Summer Vacation for Students ( F.Y.B.Tech, F.Y.M.Tech, S\nd t | 08/06/2026 to 11/07/2026\nS.Y BTech, S.Y.MTech) —\n52 | ESE Result Declaration ( F.Y.B.Tech, F.Y.M.Tech, Tae 30/06/2026\nS.Y.BTech, S.Y.MTech\n53 Commencement of S. Y. B. Tech., T. Y. B. Tech Semester —I (A. Y. 7/2026\n2026-27 | Monday 13/0'), Document(id='e2be6dd0-a1d0-4fc9-96cf-448d81b81cb5', metadata={'college_id': 'college_pict', 'source': 'academic calendar.pdf'}, page_content="*Dates will vary as per SPPU guidelines\n\nRa\n\nBoge Di ector.\n\nPrincipal\nirector\nPrinci

ValueError: too many values to unpack (expected 2)

In [16]:
import gradio as gr
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

with gr.Blocks(title="College Chatbot") as demo:
    gr.Markdown("# 🎓 College Information Chatbot")

    with gr.Row():
        college_dropdown = gr.Dropdown(
            label="Select College",
            choices=[
                "pict",
                "vit"
            ],
            value=None
        )

    chatbot = gr.Chatbot(label="Chat")
    msg = gr.Textbox(
        placeholder="Ask something about the selected college...",
        show_label=False
    )

    clear = gr.Button("Clear Chat")

    msg.submit(put_message_in_chatbot,[msg,chatbot],[msg,chatbot]).then(
        question_answer,
        inputs=[msg, chatbot, college_dropdown],
        outputs=[chatbot]
    )

    clear.click(
        lambda: [],
        outputs=[chatbot]
    )

demo.launch(inbrowser=True)



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



[Document(id='eb579e2d-c8f1-4361-aef7-77f7dadb468d', metadata={'college_id': 'college_pict', 'source': 'BNY_Mellon_Tech_Stack.pdf'}, page_content='BNY Mellon Tech Stack\nFrontend:\n- Web: React.js, Angular, HTML5, CSS3, JavaScript\n- Mobile: iOS (Swift, Objective-C), Android (Java, Kotlin)\nBackend:\n- Java, .NET (C#), Python\n- Frameworks: Spring Boot, ASP.NET Core\nDatabases:\n- Relational: Oracle, SQL Server, MySQL\n- NoSQL: MongoDB, Cassandra\n- Caching: Redis\nMessaging & Streaming:\n- Kafka, RabbitMQ, IBM MQ\nCloud & Infrastructure:\n- AWS, Microsoft Azure\n- Docker, Kubernetes\n- Terraform, Ansible\nMonitoring & Analytics:\n- Prometheus, Grafana, ELK Stack (Elasticsearch, Logstash, Kibana)\n- Splunk\nSecurity & Compliance:\n- PCI DSS, SOC2 compliant systems\n- Tokenization, Encryption protocols, IAM solutions\nDevOps & CI/CD:- Jenkins, GitLab CI/CD, Azure DevOps\n- Maven, Gradle, NuGet\nThis tech stack enables BNY Mellon to manage high-volume financial transactions securely and

c:\Users\ADMIN\Desktop\College_chatbot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:2858: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


when is the first unit test?
[Document(id='8c12b051-7889-4741-9c6b-d8e66b1ffe78', metadata={'source': 'academic calendar.pdf', 'college_id': 'college_pict'}, page_content='16/02/2026 to 18/02/2026\n18/02/2026 to 24/02/2026\n\n11__| Unit Test 1 (TE, BE) Mon to Wed\n12 | Feedback Collecti i\ne “ = Collection (Mid Sem) (S.Y.B.Tech, S.Y.M.Tech, Wed to Tue\n\n13. | Chhatrapati Shivaji Maharaj Jayanti (Holiday) Thursday 19/02/2026\n14. | PICT International Hackathon “TechFiesta 2026” Sat & Sun | 21/02/2026 & 22/02/2026\nPICTO FEST Mon to Wed | 23/02/2026 to 25/02/2026\n\nADDICTION-2026 (Annual Social Gathering) Thu to Sat | 26/02/2026 to 28/02/2026\n03/03/2026\nIn-semester Examination (ISE) ( F.Y.B.Tech, F.Y.M.Tech,\nS.Y B.Tech, S.Y.M.Tech Wed to Tue | 04/03/2026 to 10/03/2026\n19 | ISE Assessment and marks entry ( F.Y.B.Tech, F.Y.M.Tech, F\n\nS.Y.B Tech, S.Y.M.Tech Frito Sat | 06/03/2026 to 14/03/2026'), Document(id='393c6f8e-f031-4172-ab8d-172819702115', metadata={'college_id': 'college_pi

c:\Users\ADMIN\Desktop\College_chatbot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:2858: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


In [30]:
query=query_rewriting("when will collge start?",[])
query

'When will college start?'

In [12]:
def query_rewriting(query,history):
    system_msg=f'''You are an excellent query_rewriter. You are provided with the user question and the history.
    Create a standalone, explicit query.
    '''
    user_msg=f'''Question:
    {query}
    History:
    {history}
    Response only the final query.'''

    rewritten_query=llm.invoke([SystemMessage(content=system_msg),HumanMessage(content=user_msg)])
    return rewritten_query.content

    

In [ ]:
from sklearn.manifold import TSNE
collection = client.get_or_create_collection('college_pict')
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['academic', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

NameError: name 'vectors' is not defined

In [13]:
def gradio_history_to_messages(history):
    messages = []
    for his in history:
        if his['role']=='user':
            messages.append(HumanMessage(content=his['content']))
        elif his['role']=='assistant':
            messages.append(AIMessage(content=his['content']))    
    return messages 

In [46]:
history=[("hello","how you are?")]
messages=gradio_history_to_messages(history)
messages

[HumanMessage(content='hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='how you are?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
import chromadb
from chromadb.config import Settings


In [57]:
load_dotenv(override=True)
CHROMA_API=os.getenv("CHROMA_API_KEY")
client = chromadb.CloudClient(
  api_key=CHROMA_API,
  tenant='246553cc-048a-47fa-aa2a-cf9d61280656',
  database='Project'
)

In [76]:

text="""The collge authorities have decided that the college for academic year 2025-26 will start from Monday 15th July, 2025.All the students keep note of this."""
add_text(text,'pict',client)


Vector store created with 2 documents


In [33]:
from langchain_core.messages import SystemMessage, HumanMessage